In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import matplotlib
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.interpolate import CubicSpline
import matplotlib.colors as mcolors

from generate_colormap import *
from starsim_utils import *


from functools import partial

from astropy.io import fits

from scipy.optimize import curve_fit

import os
import sys

# Add the path to the correct version of your code
sys.path.insert(0, os.path.abspath("/home/sophie-stucki/starsim_david/"))
import starsim

In [2]:
def simulation_init(Q, spot_size_list, latitude_list, longitude_list, conf_file_path, periods_nbr=1, point_nbr=1, dT_fc=None):
    """
    Iniialization of the starsim simulation for a particular set of parameters

    Params:
            -Q: facular_area_ratio
    Return:
            - ss: starsim object

    """
    
    #create the starsim object
    ss=starsim.StarSim(conf_file_path=conf_file_path)

    #set timeframe
    t=np.linspace(0,ss.rotation_period*periods_nbr,int(ss.rotation_period*periods_nbr*point_nbr))

    #set the Q parameter
    ss.facular_area_ratio=Q

    if dT_fc !=None:
        ss.facula_T_contrast = dT_fc

    #initialize the spot
    overlap=True

    #TODO: more modulable
    while overlap:
        Nspots=len(spot_size_list)
        ss.spot_map=np.zeros([Nspots,7])
        for j in range(Nspots):
            ss.spot_map[j][1]=200#lifetime spot
            ss.spot_map[j][0]=0#appearance time
            ss.spot_map[j][2]=latitude_list[j]#latitude (degrees) [0,180]
            ss.spot_map[j][3]=longitude_list[j]#longitude (degrees)	[0,360]
            ss.spot_map[j][4]=spot_size_list[j]#spot size (degrees)
        overlap=starsim.nbspectra.check_spot_overlap(ss.spot_map,Q) #true if spots are overlapping
        if overlap:
            print("ERROR: Spots are overlapping")
        #checks if the are overlapping spots. If true, find another set of spot parameters

    return ss

In [3]:
conf_file_path = '/home/sophie-stucki/starsim/starsim/starsim_sun.conf'

Q_list = [0, 1e4, 2]
spot_size_list_list = [[5], [0.1], [2]]
latitude_list = [90]
longitude_list = [250]
periods_nbr = 0.8
point_nbr = 2

for i in range(len(Q_list)):
    #create the starsim object
    ss = simulation_init(Q_list[i], spot_size_list_list[i], latitude_list, longitude_list, conf_file_path, periods_nbr=periods_nbr, point_nbr=point_nbr, dT_fc=30)
    t=np.linspace(0,ss.rotation_period *periods_nbr,int(ss.rotation_period *periods_nbr*point_nbr))

    ss.compute_forward(['lc', 'rv'], t)

    np.savetxt('/home/sophie-stucki/Documents/starsim_simulations/test_active_reg_type/ff_ph_Q_{}_r_{}_lat_{}_long_{}.txt'.format(Q_list[i], spot_size_list_list[i], latitude_list, longitude_list), ss.results['ff_ph'])
    np.savetxt('/home/sophie-stucki/Documents/starsim_simulations/test_active_reg_type/ff_fc_Q_{}_r_{}_lat_{}_long_{}.txt'.format(Q_list[i], spot_size_list_list[i], latitude_list, longitude_list), ss.results['ff_fc'])
    np.savetxt('/home/sophie-stucki/Documents/starsim_simulations/test_active_reg_type/lc_Q_{}_r_{}_lat_{}_long_{}.txt'.format(Q_list[i], spot_size_list_list[i], latitude_list, longitude_list), ss.results['lc'])
    np.savetxt('/home/sophie-stucki/Documents/starsim_simulations/test_active_reg_type/rv_Q_{}_r_{}_lat_{}_long_{}.txt'.format(Q_list[i], spot_size_list_list[i], latitude_list, longitude_list), ss.results['rv'])
    np.savetxt('/home/sophie-stucki/Documents/starsim_simulations/test_active_reg_type/bis_Q_{}_r_{}_lat_{}_long_{}.txt'.format(Q_list[i], spot_size_list_list[i], latitude_list, longitude_list), ss.results['bis'])


OLD VERSION
Date 20.040000000000003. ff_ph=100.000%. ff_sp=0.000%. ff_fc=0.000%. ff_pl=0.000%. [40/40]%0 513334 0.991723131854202
1 513335 0.9953121706442206
2 513335 0.9953121706442206
3 513336 0.9984275207308333
4 513336 0.9984275207308333
5 513337 1.0010654384623798
6 513337 1.0010654384623798
7 513338 1.003343760978519
8 513339 1.005255199555457
9 513339 1.005255199555457
10 513340 1.0067067218477714
11 513340 1.0067067218477714
12 513341 1.0076568475705543
13 513341 1.0076568475705543
14 513342 1.0081676191123885
15 513342 1.0081676191123885
16 513343 1.0083603261613687
17 513344 1.0083560493590786
18 513344 1.0083560493590786
19 513345 1.0082497715005283
20 513345 1.0082497715005283
21 513346 1.0081089891235304
22 513346 1.0081089891235304
23 513347 1.0079788854265634
24 513348 1.0078870688730681
25 513348 1.0078870688730681
26 513349 1.0078481591944688
27 513349 1.0078481591944688
28 513350 1.0078644030179635
29 513350 1.0078644030179635
30 513351 1.007925678452989
31 513352 1.0

In [4]:
ss.temperature_facula

5808.0

In [5]:
array([[1.17292504e+14, 2.61435280e+13, 1.40451600e+14, ...,
        1.25653160e+14, 1.25752320e+14, 1.25643920e+14],
       [1.07382780e+14, 2.53545200e+13, 1.28581408e+14, ...,
        1.23061360e+14, 1.23144280e+14, 1.23041680e+14],
       [9.78002120e+13, 2.45860160e+13, 1.16986760e+14, ...,
        1.20367080e+14, 1.20446240e+14, 1.20343840e+14],
       ...,
       [7.22446720e+06, 2.97794080e+06, 1.07388820e+07, ...,
        2.89189200e+05, 2.89399760e+05, 2.95175880e+05],
       [3.97314000e+06, 1.90189120e+06, 6.82444480e+06, ...,
        1.57685200e+05, 1.57646360e+05, 1.63017320e+05],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])

NameError: name 'array' is not defined